Match MyNeta candidate rows to winners within the same election year and state, using normalized party and candidate name. Candidate data comes from the nine state/year Parquet files produced by data_cleaning_improved.ipynb. Exact matches are confirmed; Jaro-Winkler suggestions remain for review.

In [6]:
import pandas as pd, re, unicodedata
from rapidfuzz.distance import JaroWinkler
from pathlib import Path
import pyarrow.parquet as pq

root=Path.cwd()
candidate_files=sorted((root/"improved_cleaned_data"/"state_year").glob("*.parquet"))
if len(candidate_files)!=9: raise FileNotFoundError("Expected 9 state/year Parquet files; found "+str(len(candidate_files)))
cand=pd.concat([pd.read_parquet(path) for path in candidate_files],ignore_index=True)
win=pd.read_csv(root/"loksabha_winners_2014_2019_2024_normalized.csv",dtype=str)
cand["State_match"]=cand.State.replace({"Andhra":"Andhra Pradesh"})

def name_key(x):
    if pd.isna(x): return ""
    x=unicodedata.normalize("NFKD",str(x)).encode("ascii","ignore").decode().casefold()
    x=re.sub(r"^(?:(?:adv|dr|mr|mrs|ms|shri|smt|prof)\\.?\\s+)+","",x)
    x=re.sub(r"\\s+(?:s/o|d/o|w/o|h/o|c/o)\\.?\\s+.*$","",x)
    return re.sub(r"[^a-z0-9]+","",x)

def party_key(x):
    x=re.sub(r"[^a-z0-9]+","",str(x).casefold()) if pd.notna(x) else ""
    aliases={"bharatiyajanataparty":"bjp","ycp":"ysrcp","ysrcongressparty":"ysrcp",
    "yuvajanasramikarythucongressparty":"ysrcp","jsp":"janasenaparty",
    "rlsp":"rashtriyaloksamataparty","ljprv":"lokjanshaktipartyramvilas"}
    return aliases.get(x,x)

cand["name_key"]=cand.Candidate.map(name_key); cand["party_key"]=cand.Party.map(party_key)
win["name_key"]=win["Winner Candidate"].map(name_key); win["party_key"]=win["Winner Party"].map(party_key)

used=set(); matches=[]
for wi,w in win.iterrows():
    pool=cand[(cand["Election Year"]==int(w["Election Year"]))&(cand.State_match==w.State)&(cand.party_key==w.party_key)]
    exact=pool[pool.name_key==w.name_key]
    if len(exact)==1 and exact.index[0] not in used:
        ci=exact.index[0]; used.add(ci); matches.append((wi,ci,"exact_name_and_party",100.0,"matched_exact")); continue
    if len(exact)>1:
        matches.append((wi,None,"exact_name_and_party",100.0,"ambiguous_exact")); continue
    scores=sorted([(JaroWinkler.normalized_similarity(w.name_key,c.name_key)*100,ci) for ci,c in pool.iterrows() if ci not in used],reverse=True)
    if scores and scores[0][0]>=90 and (len(scores)==1 or scores[0][0]-scores[1][0]>=4):
        score,ci=scores[0]; used.add(ci); matches.append((wi,ci,"jaro_winkler_name_same_party",round(score,1),"review_fuzzy"))
    else:
        matches.append((wi,None,"jaro_winkler_name_same_party",round(scores[0][0],1) if scores else None,"unmatched"))

rows=[]
for wi,ci,method,score,status in matches:
    w=win.loc[wi]; c=cand.loc[ci] if ci is not None else None
    rows.append({"State":w.State,"Election Year":int(w["Election Year"]),"Constituency":w.Constituency,
    "Winner Candidate":w["Winner Candidate"],"Winner Party":w["Winner Party"],
    "Matched Candidate":c.Candidate if c is not None else None,"Candidate Party":c.Party if c is not None else None,
    "Candidate Sno":c["Sno."] if c is not None else None,"Match Method":method,
    "Name Similarity":score,"Match Status":status})
cross=pd.DataFrame(rows)
cross.to_csv(root/"winner_candidate_match_review_jw.csv",index=False,encoding="utf-8-sig")

pool=cand.drop(columns=["State_match","name_key","party_key"],errors="ignore").copy()
pool["Winner Match Status"]="Not yet matched to winner list"
for status,label in [("matched_exact","Winner: exact name and party"),("review_fuzzy","Possible winner: review fuzzy name match")]:
    for _,r in cross[cross["Match Status"]==status].dropna(subset=["Candidate Sno"]).iterrows():
        s="Andhra" if r.State=="Andhra Pradesh" else r.State
        mask=(pool.State==s)&(pool["Election Year"]==r["Election Year"])&(pool["Sno."]==r["Candidate Sno"])
        pool.loc[mask,"Winner Match Status"]=label
pool.to_csv(root/"candidate_pool_winner_match_status_jw.csv",index=False,encoding="utf-8-sig")

summary=cross.groupby(["Election Year","State","Match Status"]).size().unstack(fill_value=0)
display(summary)
print("Candidates:",len(cand),"Winners:",len(win))
print("Match totals:",cross["Match Status"].value_counts().to_dict())
print("Saved winner_candidate_match_review_jw.csv and candidate_pool_winner_match_status_jw.csv")


ArrowKeyError: A type extension with name pandas.period already defined

Review the winner crosswalk. Confirm each Jaro-Winkler suggestion against the source page or affidavit, and investigate unmatched winners. Unmatched candidates are not automatically labeled as losers; only label the remaining candidates as non-winners after accounting for every winner.